In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load and preprocess data
print("\n=== Loading Data ===")
df = pd.read_csv('user_reviews.csv')
print(f"Original dataset shape: {df.shape}")

# Remove rows with NaN in Translated_Review or Sentiment columns
print("\n=== Preprocessing ===")
df_clean = df.dropna(subset=['Translated_Review', 'Sentiment'])
print(f"After removing NaN values: {df_clean.shape}")

# Also remove rows where the string value is literally "nan" (case-insensitive)
df_clean = df_clean[
    (~df_clean['Translated_Review'].astype(str).str.lower().isin(['nan', 'none', ''])) &
    (~df_clean['Sentiment'].astype(str).str.lower().isin(['nan', 'none', '']))
]
print(f"After removing 'nan' string values: {df_clean.shape}")
print(f"Total removed: {len(df) - len(df_clean)} rows")

# Display sentiment distribution
print("\nSentiment Distribution:")
print(df_clean['Sentiment'].value_counts())

# Encode labels
label_encoder = LabelEncoder()
df_clean['label'] = label_encoder.fit_transform(df_clean['Sentiment'])
print(f"\nLabel Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# Split data (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['Translated_Review'].values,
    df_clean['label'].values,
    test_size=0.3,
    random_state=42,
    stratify=df_clean['label'].values
)

print(f"\nTrain set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Custom Dataset class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Initialize tokenizer and model
print("\n=== Initializing BERT Model ===")
try:
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    num_labels = len(label_encoder.classes_)
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )
    model = model.to(device)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nTrying alternative loading method...")
    from transformers import BertConfig
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    num_labels = len(label_encoder.classes_)
    config = BertConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
    model = BertForSequenceClassification(config)
    model = model.to(device)
    print("Model initialized with random weights!")

# Create datasets and dataloaders
train_dataset = SentimentDataset(X_train, y_train, tokenizer)
test_dataset = SentimentDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Training setup
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

# Training function
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(data_loader, desc="Training")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        progress_bar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    
    return total_loss / len(data_loader), correct / total

# Evaluation function
def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            _, preds = torch.max(outputs.logits, 1)
            
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    
    return predictions, actual_labels

# Training loop
print("\n=== Training BERT Model ===")
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")

# Evaluation
print("\n=== Evaluating on Test Set ===")
predictions, actual_labels = evaluate(model, test_loader, device)

# Calculate metrics
accuracy = accuracy_score(actual_labels, predictions)
print(f"\nTest Accuracy: {accuracy:.4f}")

print("\n=== Classification Report ===")
print(classification_report(
    actual_labels, 
    predictions, 
    target_names=label_encoder.classes_
))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(actual_labels, predictions)
print(cm)

# Save model
print("\n=== Saving Model ===")
model.save_pretrained('./app_review_sentiment_bert')
tokenizer.save_pretrained('./app_review_sentiment_bert')
print("Model saved to './app_review_sentiment_bert'")

# Example predictions
print("\n=== Example Predictions ===")
sample_texts = X_test[:5]
for i, text in enumerate(sample_texts):
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, pred = torch.max(outputs.logits, 1)
    
    predicted_sentiment = label_encoder.inverse_transform([pred.cpu().item()])[0]
    actual_sentiment = label_encoder.inverse_transform([y_test[i]])[0]
    
    print(f"\nText: {text[:100]}...")
    print(f"Predicted: {predicted_sentiment} | Actual: {actual_sentiment}")

print("\n=== Training Complete ===")

Using device: cuda

=== Loading Data ===
Original dataset shape: (64295, 5)

=== Preprocessing ===
After removing NaN values: (37427, 5)
After removing 'nan' string values: (37427, 5)
Total removed: 26868 rows

Sentiment Distribution:
Sentiment
Positive    23998
Negative     8271
Neutral      5158
Name: count, dtype: int64

Label Mapping: {'Negative': 0, 'Neutral': 1, 'Positive': 2}

Train set size: 26198
Test set size: 11229

=== Initializing BERT Model ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!

=== Training BERT Model ===

Epoch 1/3


Training: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1638/1638 [50:53<00:00,  1.86s/it, loss=0.0397, acc=0.877]


Train Loss: 0.3274, Train Accuracy: 0.8767

Epoch 2/3


...

Train Loss: 0.1298, Train Accuracy: 0.9585

Epoch 3/3


Training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1638/1638 [45:20<00:00,  1.66s/it, loss=0.00417, acc=0.975]


Train Loss: 0.0799, Train Accuracy: 0.9750

=== Evaluating on Test Set ===


Evaluating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 702/702 [02:56<00:00,  3.98it/s]



Test Accuracy: 0.9566

=== Classification Report ===
              precision    recall  f1-score   support

    Negative       0.93      0.91      0.92      2481
     Neutral       0.97      0.91      0.94      1548
    Positive       0.96      0.98      0.97      7200

    accuracy                           0.96     11229
   macro avg       0.96      0.93      0.94     11229
weighted avg       0.96      0.96      0.96     11229


=== Confusion Matrix ===
[[2266   20  195]
 [  57 1408   83]
 [ 113   19 7068]]

=== Saving Model ===
Model saved to './app_review_sentiment_bert'

=== Example Predictions ===

Text: This would great freeze time. And conveniently coincidentally tends freeze right I've activated boos...
Predicted: Positive | Actual: Positive

Text: Ok, game great. For twelve year old pretty grow up, game fine. But thing id like criticize says "EDI...
Predicted: Positive | Actual: Positive

Text: I got rip bought big box filled bracelet making materials I got small barely supp